# Student Academic Risk Early-Warning System
## Decision Support and Model Inference Demo

This notebook demonstrates direct model inference, Explainable AI factor attribution, What-If simulation, and cohort batch evaluation using `prediction_engine.py`.

In [10]:
import os
import sys
import pandas as pd

# Ensure parent project directory is in sys.path
sys.path.insert(0, os.path.abspath('..'))
from prediction_engine import engine, FEATURE_METADATA, PRESET_PERSONAS

print('Prediction engine loaded successfully.')
print('Top features:', engine.top_features)

Prediction engine loaded successfully.
Top features: ['G1', 'failures', 'absences', 'age', 'health', 'freetime', 'Walc', 'goout', 'Medu', 'famrel']


### 1. Single Student Risk Assessment

In [11]:
student_inputs = {
    'G1': 6,
    'failures': 2,
    'absences': 18,
    'age': 18,
    'health': 2,
    'freetime': 4,
    'Walc': 4,
    'goout': 4,
    'Medu': 1,
    'famrel': 2
}

result = engine.predict(student_inputs)

print('Predicted Risk Level:', result['predicted_class'], f'({result["confidence"]}%)')
print('Risk Score:', result['risk_score'], '/ 100')
print('Probabilities:', result['probabilities'])

print('\n--- Contributing Risk Drivers ---')
for d in result['contributions']['risk_drivers']:
    print(f"{d['name']}: {d['description']} (+{d['weight']}% impact)")

print('\n--- Recommended Action Plan ---')
for r in result['recommendations']:
    print(f"[{r['urgency']}] {r['action']}: {r['detail']}")

Predicted Risk Level: High (78.0%)
Risk Score: 88.7 / 100
Probabilities: {'High': 78.0, 'Medium': 21.3, 'Low': 0.7}

--- Contributing Risk Drivers ---
First Period Grade (G1): Low G1 Grade (6/20) falls below passing threshold (<= 9) (+88.4% impact)
Past Class Failures: 2 past class failures recorded (+47.4% impact)
Term Absences: 18 absences: high attendance concern (+33.9% impact)
Student Age: Age 18 years (+5.9% impact)
Weekend Alcohol Consumption: Weekend alcohol consumption (level 4/5) (+5.3% impact)
Family Relationship Quality: Low family relationship rating (2/5) (+4.9% impact)
Health Status: Low health rating (2/5) (+4.8% impact)
Mother's Education Level: Mother's education: Primary (+4.1% impact)
Free Time After School: High unstructured free time (4/5) (+3.4% impact)
Going Out with Friends: High frequency of social outings (4/5) (+2.3% impact)

--- Recommended Action Plan ---
[Urgent] Assign Peer Tutor and Remedial Review: Student has early score of 6/20 and 2 prior failure(s)

### 2. What-If Scenario Simulation

In [12]:
target_inputs = dict(student_inputs)
target_inputs['G1'] = 14
target_inputs['absences'] = 2
target_inputs['Walc'] = 1

sim = engine.simulate_what_if(student_inputs, target_inputs)
deltas = sim['deltas']

print('Category Shift:', deltas['category_shift'])
print('Risk Score Delta:', deltas['risk_score_diff'], 'points')
print('High Risk Prob Delta:', deltas['high_p_diff'], '%')

Category Shift: High Risk to Medium Risk
Risk Score Delta: -44.5 points
High Risk Prob Delta: -61.0 %


### 3. Cohort Batch Assessment

In [13]:
dataset_path = os.path.abspath('../data/raw/student-por.csv')
if os.path.exists(dataset_path):
    df_raw = pd.read_csv(dataset_path, sep=';')
    batch = engine.process_batch(df_raw.head(30))
    print('Total Students:', batch['total_students'])
    print('Counts:', batch['counts'])
    print(batch['data'][['Student ID', 'Predicted Risk', 'Confidence', 'Risk Score', 'Primary Concern']].head(10))
else:
    print('Dataset not found at:', dataset_path)

Total Students: 30
Counts: {'High': 1, 'Medium': 26, 'Low': 3}
  Student ID Predicted Risk Confidence  Risk Score  \
0    STU-001         Medium      57.6%        67.8   
1    STU-002         Medium      81.4%        56.3   
2    STU-003         Medium      93.9%        49.4   
3    STU-004            Low      61.9%        19.4   
4    STU-005         Medium      96.1%        48.5   
5    STU-006         Medium      89.3%        46.3   
6    STU-007         Medium      91.7%        47.0   
7    STU-008         Medium      87.8%        45.6   
8    STU-009            Low      73.5%        14.1   
9    STU-010         Medium      86.2%        51.7   

               Primary Concern  
0      First Period Grade (G1)  
1      First Period Grade (G1)  
2                Term Absences  
3  Family Relationship Quality  
4      First Period Grade (G1)  
5                Term Absences  
6       Free Time After School  
7      First Period Grade (G1)  
8                Health Status  
9       Free